# 🎬 OpenSource Clipping Studio — Backend Server

This notebook runs the **FastAPI backend** for OpenSource Clipping Studio.

**How it works:**
1. Run all cells below to start the server
2. Copy the **Public URL** that appears
3. Open the Studio page on GitHub Pages and paste the URL
4. Start clipping!

**Studio URL:** [naufalrizqullah.github.io/opensource-clipping/studio/](https://naufalrizqullah.github.io/opensource-clipping/studio/)

---

## 1. Setup Project

In [ ]:
# Clone repository
!rm -rf ./* ./.*
!git clone https://github.com/NaufalRizqullah/opensource-clipping.git .

In [ ]:
%%capture
# Install dependencies
!pip install -r requirements.txt
!pip install pyngrok nest-asyncio aiofiles

In [ ]:
# Setup system dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg

## 2. Configure API Keys

Make sure you have added these secrets in **Add-ons > Secrets**:
- `GOOGLE_API_KEY` — Google Gemini API key
- `NGROK_AUTHTOKEN` — ngrok auth token (get from https://dashboard.ngrok.com)
- `PEXELS_API_KEY` — (optional) for B-roll footage
- `HF_TOKEN` — (optional) for speaker diarization

In [ ]:
import os
from pathlib import Path

# Try Kaggle secrets first, then Colab
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    def get_secret(name, default=""):
        try:
            return secrets.get_secret(name) or default
        except:
            return default
    print("✅ Using Kaggle Secrets")
except ImportError:
    try:
        from google.colab import userdata
        def get_secret(name, default=""):
            try:
                return userdata.get(name) or default
            except:
                return default
        print("✅ Using Colab Secrets")
    except ImportError:
        def get_secret(name, default=""):
            return os.environ.get(name, default)
        print("⚠️ Using environment variables")

# Load secrets
GOOGLE_API_KEY = get_secret("GOOGLE_API_KEY")
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")
PEXELS_API_KEY = get_secret("PEXELS_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

# Set environment variables
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
if PEXELS_API_KEY:
    os.environ["PEXELS_API_KEY"] = PEXELS_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Create .env file
env_text = f"""# Auto-generated from notebook secrets
GOOGLE_API_KEY={GOOGLE_API_KEY}
PEXELS_API_KEY={PEXELS_API_KEY}
HF_TOKEN={HF_TOKEN}
"""
Path(".env").write_text(env_text, encoding="utf-8")

# Status
print(f"  GOOGLE_API_KEY: {'✅ Set' if GOOGLE_API_KEY else '❌ Missing'}")
print(f"  NGROK_AUTHTOKEN: {'✅ Set' if NGROK_AUTHTOKEN else '❌ Missing'}")
print(f"  PEXELS_API_KEY: {'✅ Set' if PEXELS_API_KEY else '⚪ Not set (optional)'}")
print(f"  HF_TOKEN: {'✅ Set' if HF_TOKEN else '⚪ Not set (optional)'}")

## 3. Start Backend Server + Tunnel

This cell starts the FastAPI server and creates an ngrok tunnel.

**Copy the Public URL** and paste it into the Studio page!

In [ ]:
import nest_asyncio
import threading
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

# Import the FastAPI app
from web.api.app import app

# Start FastAPI server in background thread
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True
)
server_thread.start()
print("✅ FastAPI server started on port 8000")

# Start ngrok tunnel
ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_url = ngrok.connect(8000)

print()
print("╔" + "═" * 58 + "╗")
print("║" + " 🌐 BACKEND SERVER READY!".ljust(58) + "║")
print("║" + "".ljust(58) + "║")
print("║" + f"  Public URL: {public_url.public_url}".ljust(58) + "║")
print("║" + "".ljust(58) + "║")
print("║" + "  📋 Copy the URL above and paste it into".ljust(58) + "║")
print("║" + "     the Studio 'Connect' dialog!".ljust(58) + "║")
print("║" + "".ljust(58) + "║")
print("║" + "  🔗 Studio: naufalrizqullah.github.io".ljust(58) + "║")
print("║" + "     /opensource-clipping/studio/".ljust(58) + "║")
print("╚" + "═" * 58 + "╝")
print()
print("⚠️  Keep this notebook running! Closing it will stop the server.")

## 4. Server Status Check

Run this cell anytime to verify the server is still running.

In [ ]:
import requests

try:
    res = requests.get("http://localhost:8000/api/health", timeout=5)
    health = res.json()
    print("✅ Server is running!")
    print(f"   GPU: {'✅ Available' if health.get('gpu_available') else '❌ Not available'}")
    print(f"   FFmpeg: {'✅ Available' if health.get('ffmpeg_available') else '❌ Not available'}")
    print(f"   Tunnel: {public_url.public_url}")
except Exception as e:
    print(f"❌ Server check failed: {e}")